<a href="https://colab.research.google.com/github/csabiu/Cosmology_Course/blob/main/practical/linear_structure_formation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Linear Structure Formation
### The Matter Power Spectrum and Fisher Forecasting

---

In this practical you will:
1. **Build the linear matter power spectrum** $P(k)$ from the primordial spectrum and the BBKS transfer function
2. **Use the Fisher matrix formalism** to predict how well a galaxy survey can constrain cosmological parameters

**Part 1** is hands-on: you will assemble $P(k)$ and explore its dependence on cosmological parameters (Exercises 1--2). **Part 2** is a guided tutorial walking through Fisher forecasting step by step, culminating in a design exercise (Exercise 3).

**Prerequisites:** Lecture 5 (linear perturbation theory, transfer function), Lecture 3 practical (chi-squared, likelihood).

**Exercises:** 3 exercises across 2 parts. Estimated time: 1.5 hours.

## Part 0 --- Setup

In [ ]:
!pip install -q scipy matplotlib numpy

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
from matplotlib.patches import Ellipse
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

# Nice plot defaults (matching course style)
plt.rcParams.update({
    'font.size': 13,
    'axes.labelsize': 14,
    'axes.titlesize': 15,
    'legend.fontsize': 11,
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

# ============================================================
# Planck 2018 fiducial cosmology
# ============================================================
H0    = 67.36    # km/s/Mpc
Om0   = 0.3153   # total matter density parameter
Ob0   = 0.0493   # baryon density parameter
OL0   = 1 - Om0  # cosmological constant (flat universe)
ns    = 0.9649   # scalar spectral index
As    = 2.1e-9   # primordial amplitude
sigma8_planck = 0.811  # sigma_8 from Planck
h     = H0 / 100.0
Om_h2 = Om0 * h**2

print(f"Planck 2018 fiducial: H0={H0}, Om0={Om0}, Ob0={Ob0}, OL0={OL0:.4f}")
print(f"  ns={ns}, As={As:.2e}, sigma8={sigma8_planck}")
print(f"  h={h:.4f}, Om*h^2={Om_h2:.4f}")

Planck 2018 fiducial: H0=67.36, Om0=0.3153, Ob0=0.0493, OL0=0.6847
  ns=0.9649, As=2.10e-09, sigma8=0.811
  h=0.6736, Om*h^2=0.1431


## Part 1 --- Building the Matter Power Spectrum

The linear matter power spectrum today is:

$$P(k) = A_s\, k^{n_s}\, T^2(k)$$

where $A_s$ is the primordial amplitude, $n_s$ the spectral index, and $T(k)$ the transfer function encoding sub-horizon physics (radiation pressure, free-streaming, etc.).

We use the **BBKS** (Bardeen, Bond, Kaiser, Szalay 1986) fitting formula:

$$T(q) = \frac{\ln(1 + 2.34\,q)}{2.34\,q}\left[1 + 3.89\,q + (16.1\,q)^2 + (5.46\,q)^3 + (6.71\,q)^4\right]^{-1/4}$$

where $q = k / (\Omega_m h^2)$ in units of Mpc$^{-1}$. The transfer function is $\approx 1$ on large scales ($k \ll k_{\mathrm{eq}}$) and falls off as $\sim \ln k / k^2$ on small scales, with the turnover at $k_{\mathrm{eq}} \approx \Omega_m h^2\,$Mpc$^{-1}$.

In [2]:
# ============================================================
# Provided: primordial spectrum and BBKS transfer function
# ============================================================

def P_primordial(k, As=As, ns=ns):
    """Scale-free primordial power spectrum P(k) = As * k^ns."""
    return As * k**ns

def transfer_BBKS(k, Om_h2=Om_h2):
    """BBKS transfer function (Bardeen et al. 1986).
    k in Mpc^{-1}, Om_h2 = Omega_m * h^2.
    """
    q = k / Om_h2
    q = np.atleast_1d(q).astype(float)
    T = np.ones_like(q)
    mask = q > 0
    qm = q[mask]
    T[mask] = (np.log(1 + 2.34 * qm) / (2.34 * qm) *
               (1 + 3.89 * qm + (16.1 * qm)**2 +
                (5.46 * qm)**3 + (6.71 * qm)**4)**(-0.25))
    return T.squeeze()

In [ ]:
# ============================================================
# Demo: Plot the BBKS transfer function
# ============================================================

k_demo = np.logspace(-4, 2, 500)  # Mpc^{-1}
T_demo = transfer_BBKS(k_demo)

# Estimate k_eq ~ Om_h2 (the turnover scale)
k_eq_approx = Om_h2

fig, ax = plt.subplots(figsize=(10, 6))
ax.loglog(k_demo, T_demo, 'steelblue', lw=2)
ax.axvline(k_eq_approx, color='crimson', ls='--', lw=1.5, alpha=0.7,
           label=f'$k_{{\\mathrm{{eq}}}} \\approx \\Omega_m h^2 = {k_eq_approx:.3f}$ Mpc$^{{-1}}$')
ax.set_xlabel('$k$ [Mpc$^{-1}$]')
ax.set_ylabel('$T(k)$')
ax.set_title('BBKS Transfer Function')
ax.legend()
plt.tight_layout()
plt.show()

### Exercise 1: Assemble the matter power spectrum $P(k)$

**Tasks:**
1. Define a $k$ array from $10^{-5}$ to $10$ Mpc$^{-1}$ (1000 log-spaced points)
2. Compute $P(k) = A_s\, k^{n_s}\, T^2(k)$ using the provided functions
3. Make a **2-panel figure** (14, 6):
   - **Left:** $P(k)$ on log-log axes. Annotate the peak.
   - **Right:** dimensionless power $\Delta^2(k) = k^3 P(k) / (2\pi^2)$ on log-log axes. Find and annotate the peak of $\Delta^2(k)$.
4. At what scale does $\Delta^2(k) = 1$ (onset of non-linearity)?

In [3]:
# ============================================================
# Exercise 1 --- Matter power spectrum P(k)
# ============================================================

from scipy.interpolate import interp1d


### Exercise 2: Parameter dependence of $P(k)$

**Tasks:**
1. Vary $\Omega_m h^2$ in $\{0.10,\, 0.14,\, 0.20\}$ keeping $n_s$ fixed at fiducial
2. Vary $n_s$ in $\{0.90,\, 0.9649,\, 1.05\}$ keeping $\Omega_m h^2$ fixed
3. Make a **2-panel figure** (14, 5): left panel for $\Omega_m h^2$ variation, right for $n_s$
4. Which parameter controls the **shape** (turnover position) and which controls the **tilt**?

In [4]:
# ============================================================
# Exercise 2 --- Parameter dependence of P(k)
# ============================================================


## Part 2 --- Fisher Matrix Forecasting from $P(k)$

Before building an expensive galaxy survey, we want to predict the **best possible parameter constraints** from its power spectrum measurements. The **Fisher information matrix** provides exactly this.

**Fisher matrix:** For a set of parameters $\boldsymbol{\theta}$, the Fisher matrix is:

$$F_{ij} = \sum_{\text{bins}} \frac{\partial \ln P}{\partial \theta_i}\, \frac{\partial \ln P}{\partial \theta_j}\, \frac{N_{\mathrm{eff}}(k)}{2}$$

where the effective number of modes per log-$k$ bin is:

$$N_{\mathrm{eff}}(k) = V_{\mathrm{survey}} \cdot \frac{k^3 \Delta\!\ln k}{2\pi^2} \cdot \left[\frac{P(k)}{P(k) + 1/\bar{n}}\right]^2$$

The first factor counts independent Fourier modes; the second accounts for shot noise from the finite galaxy density $\bar{n}$.

**Cramer--Rao bound:** The covariance matrix satisfies $C \geq F^{-1}$, so the best achievable marginalised uncertainty is $\sigma_i \geq \sqrt{(F^{-1})_{ii}}$. Off-diagonal elements of $F^{-1}$ encode parameter degeneracies, visualised as tilted confidence ellipses.

**Parameters:** We forecast constraints on $\boldsymbol{\theta} = \{\Omega_m h^2,\, n_s,\, \ln(10^{10}A_s)\}$ from a DESI-like galaxy survey.

In this tutorial we walk through each step. **Run each cell and read the commentary.** Exercise 3 at the end asks you to explore how survey design affects the constraints.

In [ ]:
# ============================================================
# Provided: helpers for Fisher forecasting
# ============================================================

# --- Amplitude calibration ---
# Our BBKS model gives a raw sigma_8 that differs from the Planck
# measurement.  We renormalise As so that sigma_8 = 0.811.

def W_tophat(x):
    """Fourier transform of a 3D spherical top-hat."""
    x = np.atleast_1d(x).astype(float)
    result = np.ones_like(x)
    mask = np.abs(x) > 1e-6
    xm = x[mask]
    result[mask] = 3.0 * (np.sin(xm) - xm * np.cos(xm)) / xm**3
    result[~mask] = 1.0 - np.asarray(x[~mask])**2 / 10.0
    return result.squeeze()

def sigma_R(R, As_eff=As, ns_eff=ns, Om_h2_eff=Om_h2):
    """RMS density fluctuation smoothed on scale R (h^{-1} Mpc)."""
    def integrand(lnk):
        k = np.exp(lnk)
        Pk = As_eff * k**ns_eff * transfer_BBKS(k, Om_h2=Om_h2_eff)**2
        return k**3 * Pk * W_tophat(k * R)**2 / (2 * np.pi**2)
    val, _ = quad(integrand, np.log(1e-5), np.log(10.0), limit=200)
    return np.sqrt(val)

sigma8_raw = sigma_R(8.0)
As_norm = As * (sigma8_planck / sigma8_raw)**2
print(f"Renormalised As: {As_norm:.4e}  (sigma_8 = {sigma_R(8.0, As_eff=As_norm):.4f})")

# --- Survey specification ---
survey_fiducial = {
    'name': 'DESI-like',
    'V_survey': 50.0,    # (Gpc/h)^3
    'nbar': 3e-4,        # (h/Mpc)^3
    'k_min': 0.01,       # h/Mpc
    'k_max': 0.20,       # h/Mpc
    'N_kbins': 20,
}
print("\nFiducial survey:")
for k, v in survey_fiducial.items():
    print(f"  {k}: {v}")

# --- Fisher building blocks ---
def N_eff(k, Pk, V_survey, nbar, dlnk):
    """Effective number of independent Fourier modes per k-bin."""
    V = V_survey * 1e9   # (Gpc/h)^3 -> (Mpc/h)^3
    return V * k**3 * dlnk / (2 * np.pi**2) * (Pk / (Pk + 1.0 / nbar))**2

def Pk_model(k, omh2, n_s, ln10As):
    """P(k) in (Mpc/h)^3 for k in h/Mpc."""
    A = np.exp(ln10As) / 1e10
    k_Mpc = k * h
    return A * k_Mpc**n_s * transfer_BBKS(k_Mpc, omh2)**2 * h**3

def build_fisher(survey, theta_fid, eps_frac=0.01):
    """Build the Fisher matrix for a given survey and fiducial parameters.
    Returns F, C, sigma_marg, k_c, dlogP, Pk_fid.
    """
    n_p = len(theta_fid)
    k_edges = np.logspace(np.log10(survey['k_min']),
                           np.log10(survey['k_max']),
                           survey['N_kbins'] + 1)
    k_c = np.sqrt(k_edges[:-1] * k_edges[1:])
    dlnk = np.log(k_edges[1:] / k_edges[:-1])
    Pk_fid = Pk_model(k_c, *theta_fid)

    # Log-derivatives
    dlogP = np.zeros((n_p, len(k_c)))
    for i in range(n_p):
        eps = eps_frac * np.abs(theta_fid[i])
        tp, tm = theta_fid.copy(), theta_fid.copy()
        tp[i] += eps; tm[i] -= eps
        dlogP[i] = (np.log(Pk_model(k_c, *tp)) -
                     np.log(Pk_model(k_c, *tm))) / (2 * eps)

    # Fisher matrix
    Neff = N_eff(k_c, Pk_fid, survey['V_survey'], survey['nbar'], dlnk)
    F = np.zeros((n_p, n_p))
    for a in range(len(k_c)):
        for i in range(n_p):
            for j in range(n_p):
                F[i, j] += dlogP[i, a] * dlogP[j, a] * Neff[a] / 2.0

    C = np.linalg.inv(F)
    sigma_marg = np.sqrt(np.diag(C))
    return F, C, sigma_marg, k_c, dlogP, Pk_fid

def draw_ellipse(ax, cov_2x2, mean, color='steelblue', label=None, levels=[1, 2]):
    """Draw confidence ellipses from a 2x2 covariance matrix."""
    eigvals, eigvecs = np.linalg.eigh(cov_2x2)
    angle = np.degrees(np.arctan2(eigvecs[1, 0], eigvecs[0, 0]))
    dchi2 = {1: 2.30, 2: 6.17}
    for lev in levels:
        w = 2 * np.sqrt(eigvals[0] * dchi2[lev])
        hv = 2 * np.sqrt(eigvals[1] * dchi2[lev])
        ell = Ellipse(xy=mean, width=w, height=hv, angle=angle,
                      fc=color, alpha=0.15 if lev == 2 else 0.3,
                      ec=color, lw=1.5,
                      label=label if lev == levels[0] else None)
        ax.add_patch(ell)

### Step 1 --- Sensitivity of $P(k)$ to parameters

The Fisher matrix requires derivatives $\partial \ln P / \partial \theta_i$. We compute these numerically via central finite differences with step size $\varepsilon = 0.01\,|\theta_i|$. The shape of each derivative tells us where in $k$-space a parameter has the most leverage.

In [ ]:
# ============================================================
# Tutorial: log-derivatives of P(k)
# ============================================================

theta_names = [r'$\Omega_m h^2$', r'$n_s$', r'$\ln(10^{10}A_s)$']
theta_fid = np.array([Om_h2, ns, np.log(1e10 * As_norm)])

# Use build_fisher to get derivatives
F_fid, C_fid, sigma_fid, k_c, dlogP, Pk_fid = build_fisher(survey_fiducial, theta_fid)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['steelblue', 'darkorange', 'seagreen']
for i, (name, col) in enumerate(zip(theta_names, colors)):
    ax.semilogx(k_c, dlogP[i], color=col, lw=2, label=name)
ax.axhline(0, color='k', ls=':', lw=0.8)
ax.set_xlabel(r'$k$ [$h\,\mathrm{Mpc}^{-1}$]')
ax.set_ylabel(r'$\partial \ln P / \partial \theta_i$')
ax.set_title('How sensitive is $P(k)$ to each parameter?')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Note: d ln P / d ln(10^10 As) = {dlogP[2].mean():.3f} (scale-independent, as expected)")

### Step 2 --- The Fisher matrix and parameter constraints

We sum the derivative products weighted by the number of modes in each bin. Inverting gives the covariance matrix, whose diagonal entries are the best-achievable variances (the Cramer--Rao bound).

In [ ]:
# ============================================================
# Tutorial: Fisher matrix constraints
# ============================================================

print("Fisher matrix F:")
for row in F_fid:
    print("  [" + "  ".join(f"{v:12.1f}" for v in row) + "]")

eigvals = np.linalg.eigvalsh(F_fid)
print(f"\nEigenvalues: {eigvals}  (all positive)")

print(f"\nMarginalised 1-sigma constraints ({survey_fiducial['name']}):")
print(f"{'Parameter':<25} {'Fiducial':>10} {'sigma':>10} {'%':>8}")
print("-" * 55)
for i, name in enumerate(theta_names):
    pct = 100 * sigma_fid[i] / np.abs(theta_fid[i])
    print(f"  {name:<23} {theta_fid[i]:10.4f} {sigma_fid[i]:10.5f} {pct:7.1f}%")

### Step 3 --- Confidence ellipses

The 2D projections of the covariance matrix show parameter degeneracies as tilted ellipses. A tilted ellipse means the two parameters are correlated --- the survey constrains a specific combination better than either individually.

In [ ]:
# ============================================================
# Tutorial: confidence ellipses for the fiducial survey
# ============================================================

pairs = [(0, 1), (0, 2), (1, 2)]
pair_labels = [
    (r'$\Omega_m h^2$', r'$n_s$'),
    (r'$\Omega_m h^2$', r'$\ln(10^{10}A_s)$'),
    (r'$n_s$', r'$\ln(10^{10}A_s)$'),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (i, j), (xlab, ylab) in zip(axes, pairs, pair_labels):
    cov_2x2 = C_fid[np.ix_([i, j], [i, j])]
    mean = theta_fid[[i, j]]
    draw_ellipse(ax, cov_2x2, mean, color='steelblue', label=survey_fiducial['name'])
    ax.plot(*mean, 'k+', ms=12, mew=2)
    ax.set_xlabel(xlab); ax.set_ylabel(ylab)
    ax.legend(fontsize=10)
    sx, sy = np.sqrt(cov_2x2[0, 0]), np.sqrt(cov_2x2[1, 1])
    ax.set_xlim(mean[0] - 4*sx, mean[0] + 4*sx)
    ax.set_ylim(mean[1] - 4*sy, mean[1] + 4*sy)
fig.suptitle('Fisher Forecast: DESI-like Survey', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

### Exercise 3: How does survey design affect constraints?

Now it is your turn. Define three survey configurations that differ from the fiducial and compare the resulting Fisher constraints.

**Tasks:**
1. Define three modified surveys:
   - **Shallow survey:** $V = 10\,(\mathrm{Gpc}/h)^3$, same $\bar{n}$ and $k_{\max}$
   - **Dense survey:** $\bar{n} = 10^{-3}\,(h/\mathrm{Mpc})^3$, same $V$ and $k_{\max}$
   - **Extended $k_{\max}$:** $k_{\max} = 0.30\,h/\mathrm{Mpc}$, same $V$ and $\bar{n}$
2. Use `build_fisher()` for each to get constraints
3. Make a **3-panel figure** (same layout as Step 3) overlaying the $1\sigma$ ellipses from **all four surveys** (fiducial + 3 variants) in different colours
4. Print a summary table comparing marginalised $\sigma_i$ across all surveys
5. **Answer:**
   - (a) Which design change helps most for $\Omega_m h^2$? For $n_s$?
   - (b) Why does increasing $\bar{n}$ help less on large scales?
   - (c) Why should we be cautious about pushing $k_{\max}$ beyond $0.2\,h/\mathrm{Mpc}$?

In [5]:
# ============================================================
# Exercise 3 --- Survey design comparison
# ============================================================


## Summary

In this practical you have:

- **Built the linear matter power spectrum** $P(k) = A_s\, k^{n_s}\, T^2(k)$ using the BBKS transfer function
- **Explored how $\Omega_m h^2$ shifts the turnover** while $n_s$ tilts the spectrum
- **Walked through the Fisher matrix formalism:** derivatives $\to$ Fisher matrix $\to$ covariance $\to$ ellipses
- **Investigated how survey volume, galaxy density, and $k_{\max}$** affect forecasted constraints
